<a href="https://colab.research.google.com/github/hrishikeshkhade/Verification/blob/main/PART4Q3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
!apt-get update -qq
!apt-get install -y cbmc

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
cbmc is already the newest version (5.95.1-4ubuntu1).
0 upgraded, 0 newly installed, 0 to remove and 38 not upgraded.


In [12]:
%%writefile one.c
// LI1 (bounds/typing invariant): 0 <= low <= high <= len
#define len 4
#include <stdbool.h>
_Bool nondet_bool();
unsigned int nondet_uint();
int nondet_int();

int main() {
   int i,j,k,x;
   int myarr[len];
   unsigned low, high, middle;
   bool exit;

   for(j=0; j<len-1; j++) { __CPROVER_assume(myarr[j] > myarr[j+1]); }

   // ---------- INITIATION ----------
   low=0; high=len; exit=0; i=-1;
   __CPROVER_assert( low<=high && high<=len, "INIT: LI1 holds before loop" );

   // ---------- CONSECUTION ----------
   i    = nondet_int();
   low  = nondet_uint();
   high = nondet_uint();
   exit = nondet_bool();
   __CPROVER_assume( low<=high && high<=len );  // assume candidate LI1 (this alone bounds 'middle' safely, no extra scaffolding needed)

   if(!exit && low<high) {
      middle = low + ((high-low)>>1);
      if(myarr[middle]<x)      high = middle;
      else if(myarr[middle]>x) low  = middle+1;
      else                     { i = middle; exit = 1; }

      __CPROVER_assert( low<=high && high<=len, "CONSEC: LI1 preserved by loop body" );
   }
   return 0;
}

Overwriting one.c


In [13]:
!cbmc one.c --unwind 4 --unwinding-assertions



CBMC version 5.95.1 (cbmc-5.95.1) 64-bit x86_64 linux
Parsing one.c
Converting
Type-checking one
Generating GOTO Program
Adding CPROVER library (x86_64)
Removal of function pointers and virtual functions
Generic Property Instrumentation
Running with 8 object bits, 56 offset bits (default)
Starting Bounded Model Checking
Unwinding loop main.0 iteration 1 file one.c line 14 function main thread 0
Unwinding loop main.0 iteration 2 file one.c line 14 function main thread 0
Unwinding loop main.0 iteration 3 file one.c line 14 function main thread 0
Runtime Symex: 0.00275355s
size of program expression: 77 steps
simple slicing removed 10 assignments
Generated 2 VCC(s), 1 remaining after simplification
Runtime Postprocess Equation: 2.2424e-05s
Passing problem to propositional reduction
converting SSA
Runtime Convert SSA: 0.00189999s
Running propositional reduction
Post-processing
Runtime Post-process: 5.456e-06s
Solving with MiniSAT 2.2.1 with simplifier
1505 variables, 3933 clauses
SAT check

In [14]:
%%writefile two.c
// LI2 (left-exclusion invariant): (low == 0) || (myarr[low-1] > x)
// i.e. everything strictly left of the current window has already been
// ruled out because it is provably bigger than x.
#define len 4
#include <stdbool.h>
_Bool nondet_bool();
unsigned int nondet_uint();
int nondet_int();

int main() {
   int i,j,k,x;
   int myarr[len];
   unsigned low, high, middle;
   bool exit;

   for(j=0; j<len-1; j++) { __CPROVER_assume(myarr[j] > myarr[j+1]); }

   // ---------- INITIATION ----------
   low=0; high=len; exit=0; i=-1;
   __CPROVER_assert( (low==0) || (myarr[low-1] > x), "INIT: LI2 holds before loop" );

   // ---------- CONSECUTION ----------
   i    = nondet_int();
   low  = nondet_uint();
   high = nondet_uint();
   exit = nondet_bool();
   __CPROVER_assume( low<=high && high<=len );          // LI1 as scaffolding (bounds 'low' so myarr[low-1] is safe)
   __CPROVER_assume( (low==0) || (myarr[low-1] > x) );  // assume candidate LI2

   if(!exit && low<high) {
      middle = low + ((high-low)>>1);
      if(myarr[middle]<x)      high = middle;
      else if(myarr[middle]>x) low  = middle+1;
      else                     { i = middle; exit = 1; }

      __CPROVER_assert( (low==0) || (myarr[low-1] > x), "CONSEC: LI2 preserved by loop body" );
   }
   return 0;
}

Overwriting two.c


In [15]:
!cbmc two.c --unwind 4 --unwinding-assertions


CBMC version 5.95.1 (cbmc-5.95.1) 64-bit x86_64 linux
Parsing two.c
Converting
Type-checking two
Generating GOTO Program
Adding CPROVER library (x86_64)
Removal of function pointers and virtual functions
Generic Property Instrumentation
Running with 8 object bits, 56 offset bits (default)
Starting Bounded Model Checking
Unwinding loop main.0 iteration 1 file two.c line 16 function main thread 0
Unwinding loop main.0 iteration 2 file two.c line 16 function main thread 0
Unwinding loop main.0 iteration 3 file two.c line 16 function main thread 0
Runtime Symex: 0.00301362s
size of program expression: 78 steps
simple slicing removed 10 assignments
Generated 2 VCC(s), 1 remaining after simplification
Runtime Postprocess Equation: 2.0042e-05s
Passing problem to propositional reduction
converting SSA
Runtime Convert SSA: 0.00263874s
Running propositional reduction
Post-processing
Runtime Post-process: 5.53e-06s
Solving with MiniSAT 2.2.1 with simplifier
1975 variables, 6074 clauses
SAT checke

In [16]:
%%writefile three.c
// LI3 (right-exclusion invariant): (high == len) || (myarr[high] < x)
// i.e. everything at/after the current window has already been ruled out
// because it is provably smaller than x.
#define len 4
#include <stdbool.h>
_Bool nondet_bool();
unsigned int nondet_uint();
int nondet_int();

int main() {
   int i,j,k,x;
   int myarr[len];
   unsigned low, high, middle;
   bool exit;

   for(j=0; j<len-1; j++) { __CPROVER_assume(myarr[j] > myarr[j+1]); }

   // ---------- INITIATION ----------
   low=0; high=len; exit=0; i=-1;
   __CPROVER_assert( (high==len) || (myarr[high] < x), "INIT: LI3 holds before loop" );

   // ---------- CONSECUTION ----------
   i    = nondet_int();
   low  = nondet_uint();
   high = nondet_uint();
   exit = nondet_bool();
   __CPROVER_assume( low<=high && high<=len );           // LI1 as scaffolding (bounds 'high' so myarr[high] is safe)
   __CPROVER_assume( (high==len) || (myarr[high] < x) ); // assume candidate LI3

   if(!exit && low<high) {
      middle = low + ((high-low)>>1);
      if(myarr[middle]<x)      high = middle;
      else if(myarr[middle]>x) low  = middle+1;
      else                     { i = middle; exit = 1; }

      __CPROVER_assert( (high==len) || (myarr[high] < x), "CONSEC: LI3 preserved by loop body" );
   }
   return 0;
}

Overwriting three.c


In [17]:
!cbmc three.c --unwind 4 --unwinding-assertions


CBMC version 5.95.1 (cbmc-5.95.1) 64-bit x86_64 linux
Parsing three.c
Converting
Type-checking three
Generating GOTO Program
Adding CPROVER library (x86_64)
Removal of function pointers and virtual functions
Generic Property Instrumentation
Running with 8 object bits, 56 offset bits (default)
Starting Bounded Model Checking
Unwinding loop main.0 iteration 1 file three.c line 16 function main thread 0
Unwinding loop main.0 iteration 2 file three.c line 16 function main thread 0
Unwinding loop main.0 iteration 3 file three.c line 16 function main thread 0
Runtime Symex: 0.00318043s
size of program expression: 78 steps
simple slicing removed 10 assignments
Generated 2 VCC(s), 1 remaining after simplification
Runtime Postprocess Equation: 2.2461e-05s
Passing problem to propositional reduction
converting SSA
Runtime Convert SSA: 0.00256472s
Running propositional reduction
Post-processing
Runtime Post-process: 5.221e-06s
Solving with MiniSAT 2.2.1 with simplifier
1851 variables, 5640 clauses

In [18]:
%%writefile four.c
// LI4 (exit/i consistency invariant): exit  <=>  i >= 0
// exit becomes true in exactly the same step that i is set to a found index.
#define len 4
#include <stdbool.h>
_Bool nondet_bool();
unsigned int nondet_uint();
int nondet_int();

int main() {
   int i,j,k,x;
   int myarr[len];
   unsigned low, high, middle;
   bool exit;

   for(j=0; j<len-1; j++) { __CPROVER_assume(myarr[j] > myarr[j+1]); }

   // ---------- INITIATION ----------
   low=0; high=len; exit=0; i=-1;
   __CPROVER_assert( (exit==1) == (i>=0), "INIT: LI4 holds before loop" );

   // ---------- CONSECUTION ----------
   i    = nondet_int();
   low  = nondet_uint();
   high = nondet_uint();
   exit = nondet_bool();
   __CPROVER_assume( low<=high && high<=len );  // basic domain bound
   __CPROVER_assume( (exit==1) == (i>=0) );     // assume candidate LI4

   if(!exit && low<high) {
      middle = low + ((high-low)>>1);
      if(myarr[middle]<x)      high = middle;
      else if(myarr[middle]>x) low  = middle+1;
      else                     { i = middle; exit = 1; }

      __CPROVER_assert( (exit==1) == (i>=0), "CONSEC: LI4 preserved by loop body" );
   }
   return 0;
}

Overwriting four.c


In [19]:
!cbmc three.c --unwind 4 --unwinding-assertions


CBMC version 5.95.1 (cbmc-5.95.1) 64-bit x86_64 linux
Parsing three.c
Converting
Type-checking three
Generating GOTO Program
Adding CPROVER library (x86_64)
Removal of function pointers and virtual functions
Generic Property Instrumentation
Running with 8 object bits, 56 offset bits (default)
Starting Bounded Model Checking
Unwinding loop main.0 iteration 1 file three.c line 16 function main thread 0
Unwinding loop main.0 iteration 2 file three.c line 16 function main thread 0
Unwinding loop main.0 iteration 3 file three.c line 16 function main thread 0
Runtime Symex: 0.00298191s
size of program expression: 78 steps
simple slicing removed 10 assignments
Generated 2 VCC(s), 1 remaining after simplification
Runtime Postprocess Equation: 2.2641e-05s
Passing problem to propositional reduction
converting SSA
Runtime Convert SSA: 0.00316395s
Running propositional reduction
Post-processing
Runtime Post-process: 7.03e-06s
Solving with MiniSAT 2.2.1 with simplifier
1851 variables, 5640 clauses
